# LSTM 情感分类

使用与 TextCNN 相同的数据划分和序列处理设置，先搭建基础 LSTM 分类器并检查前向计算。测试集留到最终评估。

In [ ]:
# 加载训练集和验证集
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader

from sentiment_analysis.paths import CLEANED_DIR
from sentiment_analysis.sequence_data import build_vocabulary, SentimentDataset

train_data = pd.read_csv(CLEANED_DIR / "train.csv")
validation_data = pd.read_csv(CLEANED_DIR / "val.csv")

print(f"Train: {len(train_data):,}")
print(f"Validation: {len(validation_data):,}")

In [ ]:
# 使用与 TextCNN 相同的序列处理配置
max_words = 20_000
max_length = 512
batch_size = 32
random_seed = 42

vocabulary = build_vocabulary(train_data["text"], max_words=max_words)
train_dataset = SentimentDataset(train_data, vocabulary, max_length)
validation_dataset = SentimentDataset(validation_data, vocabulary, max_length)

generator = torch.Generator().manual_seed(random_seed)
train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, generator=generator,
)
validation_loader = DataLoader(
    validation_dataset, batch_size=batch_size, shuffle=False,
)

print(f"Vocabulary size: {len(vocabulary):,}")
print(f"Train batches: {len(train_loader):,}")
print(f"Validation batches: {len(validation_loader):,}")

## 基础 LSTM 分类器

Embedding 将 Token ID 转换为词向量；LSTM 按词序处理评论；最后使用每条评论的最终有效隐藏状态进行分类。通过有效长度跳过末尾填充。

In [ ]:
# 定义单层、单向 LSTM 情感分类器
class LSTMClassifier(nn.Module):
    def __init__(self, vocabulary_size: int, embedding_dim: int, hidden_size: int,
                 padding_index: int, dropout: float = 0.5):
        super().__init__()

        self.embedding = nn.Embedding(vocabulary_size, embedding_dim, padding_idx=padding_index)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, token_ids, lengths):
        if (lengths <= 0).any():
            raise ValueError("Each sequence must contain at least one token.")

        embeddings = self.embedding(token_ids)
        packed_embeddings = pack_padded_sequence(
            embeddings, lengths.cpu(), batch_first=True, enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed_embeddings)
        final_hidden = hidden[-1]

        return self.classifier(self.dropout(final_hidden))

In [ ]:
# 检查一个批次的前向计算
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(random_seed)

lstm_model = LSTMClassifier(
    vocabulary_size=len(vocabulary),
    embedding_dim=128,
    hidden_size=128,
    padding_index=vocabulary["<PAD>"],
).to(device)

batch_token_ids, batch_lengths, batch_labels = next(iter(train_loader))
batch_token_ids = batch_token_ids.to(device)
batch_labels = batch_labels.to(device)

lstm_model.eval()
with torch.no_grad():
    batch_logits = lstm_model(batch_token_ids, batch_lengths)
    batch_loss = nn.CrossEntropyLoss()(batch_logits, batch_labels)

print("Device:", device)
print("Token IDs shape:", batch_token_ids.shape)
print("Output shape:", batch_logits.shape)
print(f"Batch loss: {batch_loss.item():.4f}")

# 后续训练前重新初始化模型和打乱顺序
generator.manual_seed(random_seed)

## LSTM 训练与验证

重新初始化模型，最多训练 20 轮；按验证损失保存最佳模型，连续 4 轮没有改善时提前停止。训练时使用梯度裁剪，测试集不参与模型选择。

In [ ]:
# 定义训练与验证流程
from sentiment_analysis.evaluation import classification_metrics

def train_one_epoch(model, data_loader, optimizer, loss_function, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for token_ids, lengths, labels in data_loader:
        token_ids = token_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(token_ids, lengths)
        loss = loss_function(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate_model(model, data_loader, loss_function, device):
    model.eval()
    total_loss = 0.0
    total = 0
    y_true = []
    y_prediction = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            token_ids = token_ids.to(device)
            labels = labels.to(device)

            logits = model(token_ids, lengths)
            loss = loss_function(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total += labels.size(0)
            y_true.extend(labels.cpu().tolist())
            y_prediction.extend(logits.argmax(dim=1).cpu().tolist())

    return total_loss / total, classification_metrics(y_true, y_prediction)

In [ ]:
# 初始化训练模型、优化器和早停配置
from sentiment_analysis.lstm import LSTMClassifier as TrainingLSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, TABLES_DIR
from sentiment_analysis.sequence_data import TOKEN_PATTERN

model_config = {
    "vocabulary_size": len(vocabulary),
    "embedding_dim": 128,
    "hidden_size": 128,
    "padding_index": vocabulary["<PAD>"],
    "dropout": 0.5,
}

torch.manual_seed(random_seed)
generator.manual_seed(random_seed)

lstm_model = TrainingLSTMClassifier(**model_config).to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

max_epochs = 20
patience = 4
best_validation_loss = float("inf")
epochs_without_improvement = 0
training_history = []

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINTS_DIR / "lstm_best.pt"

In [ ]:
# 训练并按验证损失保存最佳模型
for epoch in range(1, max_epochs + 1):
    train_loss, train_accuracy = train_one_epoch(
        lstm_model, train_loader, optimizer, loss_function, device,
    )
    validation_loss, validation_metrics = evaluate_model(
        lstm_model, validation_loader, loss_function, device,
    )

    training_history.append({
        "EPOCH": epoch,
        "TRAIN LOSS": train_loss,
        "TRAIN ACCURACY": train_accuracy,
        "VALIDATION LOSS": validation_loss,
        "VALIDATION ACCURACY": validation_metrics["ACCURACY"],
        "VALIDATION F1": validation_metrics["F1"],
    })
    training_results = pd.DataFrame(training_history)
    training_results.to_csv(TABLES_DIR / "lstm_training_history.csv", index=False)

    print(
        f"Epoch {epoch}: Train loss {train_loss:.4f}, accuracy {train_accuracy:.4f} | "
        f"Validation loss {validation_loss:.4f}, accuracy {validation_metrics['ACCURACY']:.4f}, "
        f"F1 {validation_metrics['F1']:.4f}", flush=True,
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": lstm_model.state_dict(),
            "model_config": model_config,
            "vocabulary": vocabulary,
            "max_length": max_length,
            "token_pattern": TOKEN_PATTERN.pattern,
            "lowercase": True,
            "random_seed": random_seed,
            "epoch": epoch,
            "validation_loss": validation_loss,
        }, checkpoint_path)
        print(f"Best model saved: {checkpoint_path}", flush=True)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print("Early stopping: validation loss did not improve.", flush=True)
            break

display(training_results.round(4))

In [ ]:
# 绘制训练与验证曲线
import matplotlib.pyplot as plt
from sentiment_analysis.paths import FIGURES_DIR

figure, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(training_results["EPOCH"], training_results["TRAIN LOSS"], marker="o", label="Train")
axes[0].plot(training_results["EPOCH"], training_results["VALIDATION LOSS"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(training_results["EPOCH"], training_results["TRAIN ACCURACY"], marker="o", label="Train")
axes[1].plot(training_results["EPOCH"], training_results["VALIDATION ACCURACY"], marker="o", label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].legend()

figure.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "lstm_training_curves.png", dpi=150)
plt.show()